# Chapter 9: Unsupervised Learning

```{admonition} Learning Objectives
:class: tip
- Understand clustering algorithms (k-Means, hierarchical, DBSCAN)
- Apply dimensionality reduction (PCA, SVD, NMF)
- Master the EM algorithm
- Build autoencoders for representation learning
- Compare different unsupervised methods
- Apply unsupervised learning to real problems
```

```{epigraph}
Unsupervised learning models common interrelationships and patterns among data attributes, discovering structure without explicit labels.

-- Charu Aggarwal
```

## 9.1 Introduction

**Unsupervised learning** discovers patterns in data without labeled examples.

### Supervised vs Unsupervised

| Aspect | Supervised | Unsupervised |
|--------|-----------|--------------|
| Data | Labeled $(\mathbf{x}, y)$ pairs | Unlabeled $\mathbf{x}$ only |
| Goal | Predict labels | Find structure |
| Output | Classification/regression model | Clusters/representations |
| Examples | Spam detection, face recognition | Customer segmentation, compression |

### Main Tasks

**1. Clustering**:
- Group similar data points
- Applications: Customer segmentation, document organization

**2. Dimensionality Reduction**:
- Find low-dimensional representations
- Applications: Visualization, preprocessing, compression

**3. Density Estimation**:
- Model probability distribution of data
- Applications: Anomaly detection, generation

### Why Unsupervised Learning?

1. **Labels are expensive**: Manual labeling is time-consuming
2. **Discover hidden structure**: Find patterns humans might miss
3. **Preprocessing**: Create features for supervised learning
4. **Compression**: Reduce data size while preserving information
5. **Anomaly detection**: Identify unusual patterns

## 9.2 k-Means Clustering

**k-Means** partitions $n$ data points into $k$ clusters by minimizing within-cluster variance.

### 9.2.1 Problem Formulation

**Objective**: Minimize sum of squared distances to cluster centers

$$\min_{\mathbf{Y}_1,...,\mathbf{Y}_k} \sum_{i=1}^{n} \min_j ||\mathbf{X}_i - \mathbf{Y}_j||^2$$

where:
- $\mathbf{X}_i$: $i$-th data point
- $\mathbf{Y}_j$: $j$-th cluster centroid
- $k$: number of clusters (user-specified)

**Challenge**: Both cluster assignments and centroids are unknown

**Solution**: Alternating optimization (iterative refinement)

### 9.2.2 k-Means Algorithm

```
Algorithm: k-MEANS(Data D, num_clusters k, max_iterations T)

begin
    // Step 1: Initialize centroids
    Y ← Randomly sample k points from D
    // Alternative: k-means++ initialization
    
    for t = 1 to T do
        // Step 2: Assign each point to nearest centroid
        for i = 1 to n do
            C[i] ← argmin_j ||X_i - Y_j||²
        
        // Step 3: Update centroids
        for j = 1 to k do
            Y_j ← mean of all X_i where C[i] = j
        
        // Check convergence
        if centroids unchanged then
            break
    
    return C, Y  // Cluster assignments and centroids
end
```

### 9.2.3 Complexity Analysis

**Time per iteration**: $O(nkd)$
- $n$ points × $k$ clusters × $d$ dimensions

**Space**: $O(n + kd)$
- Store assignments and centroids

**Convergence**: Typically $O(\log n)$ iterations

**Total**: $O(nkd \log n)$

### 9.2.4 Properties

**Advantages**:
- Simple and fast
- Scales to large datasets
- Easy to implement

**Disadvantages**:
- Requires $k$ to be specified
- Sensitive to initialization
- Assumes spherical clusters
- Sensitive to outliers
- Converges to local optimum

### 9.2.5 Initialization: k-Means++

**Better initialization** to avoid poor local optima:

```
Algorithm: k-MEANS++ Initialization

begin
    // Choose first centroid randomly
    Y_1 ← Random point from D
    
    for j = 2 to k do
        // For each point, compute distance to nearest centroid
        for i = 1 to n do
            d_i ← min_{j' < j} ||X_i - Y_{j'}||²
        
        // Choose next centroid with probability ∝ d_i²
        Y_j ← Sample from D with probability d_i² / ∑ d_i²
    
    return Y
end
```

**Improvement**: Provably better than random initialization

### 9.2.6 Choosing k: Elbow Method

**Within-cluster sum of squares (WCSS)**:
$$\text{WCSS}(k) = \sum_{j=1}^{k} \sum_{\mathbf{X}_i \in C_j} ||\mathbf{X}_i - \mathbf{Y}_j||^2$$

**Method**:
1. Run k-means for $k = 1, 2, ..., K$
2. Plot WCSS vs $k$
3. Look for "elbow" where decrease slows

**Note**: WCSS always decreases with $k$, but elbow indicates diminishing returns

## 9.3 Hierarchical Clustering

**Hierarchical clustering** creates tree-like hierarchy of clusters (dendrogram).

### 9.3.1 Types

**1. Agglomerative (Bottom-Up)**:
- Start with each point as cluster
- Iteratively merge closest clusters
- Most common approach

**2. Divisive (Top-Down)**:
- Start with all points in one cluster
- Iteratively split clusters
- Less common

### 9.3.2 Agglomerative Clustering

```
Algorithm: AGGLOMERATIVE-CLUSTERING(Data D, num_clusters k)

begin
    // Step 1: Initialize each point as cluster
    C ← {{X_1}, {X_2}, ..., {X_n}}
    
    // Step 2: Compute pairwise distances
    M ← n×n distance matrix between clusters
    
    // Step 3: Iteratively merge closest clusters
    while |C| > k do
        // Find closest pair
        (i, j) ← argmin_{i<j} M[i,j]
        
        // Merge clusters
        C_new ← C_i ∪ C_j
        C ← (C \ {C_i, C_j}) ∪ {C_new}
        
        // Update distance matrix
        for each cluster C_m ≠ C_new do
            M[new, m] ← LINKAGE(C_new, C_m)
        
        Remove rows/columns i, j from M
    
    return C
end
```

### 9.3.3 Linkage Criteria

**How to measure distance between clusters?**

For clusters $C_i$ and $C_j$:

**1. Single Linkage** (minimum):
$$d(C_i, C_j) = \min_{\mathbf{x} \in C_i, \mathbf{y} \in C_j} ||\mathbf{x} - \mathbf{y}||$$

- **Pros**: Can find non-spherical clusters
- **Cons**: Sensitive to noise (chaining effect)

**2. Complete Linkage** (maximum):
$$d(C_i, C_j) = \max_{\mathbf{x} \in C_i, \mathbf{y} \in C_j} ||\mathbf{x} - \mathbf{y}||$$

- **Pros**: Creates compact clusters
- **Cons**: Sensitive to outliers

**3. Average Linkage**:
$$d(C_i, C_j) = \frac{1}{|C_i||C_j|} \sum_{\mathbf{x} \in C_i} \sum_{\mathbf{y} \in C_j} ||\mathbf{x} - \mathbf{y}||$$

- **Pros**: Balanced, less sensitive to outliers
- **Cons**: Moderate computational cost

**4. Ward's Linkage** (minimum variance):
$$d(C_i, C_j) = \frac{|C_i||C_j|}{|C_i| + |C_j|} ||\boldsymbol{\mu}_i - \boldsymbol{\mu}_j||^2$$

where $\boldsymbol{\mu}_i$ is mean of $C_i$.

- **Pros**: Minimizes within-cluster variance
- **Cons**: Assumes spherical clusters

### 9.3.4 Complexity

**Time**: $O(n^3)$ for naive implementation
- $O(n^2)$ iterations, $O(n)$ per iteration

**Optimized**: $O(n^2 \log n)$ with priority queues

**Space**: $O(n^2)$ for distance matrix

### 9.3.5 Dendrogram

**Tree structure** showing merge hierarchy:
- Leaves: Individual points
- Internal nodes: Merged clusters
- Height: Distance at which merge occurred

**Use**: Cut dendrogram at desired height to get $k$ clusters

## 9.4 DBSCAN

**DBSCAN** (Density-Based Spatial Clustering) finds clusters of arbitrary shape based on density.

### 9.4.1 Key Concepts

**Parameters**:
- $\epsilon$: Neighborhood radius
- MinPts: Minimum points to form dense region

**Point Types**:

**1. Core Point**: Has ≥ MinPts points within $\epsilon$
$$|N_\epsilon(\mathbf{x})| \geq \text{MinPts}$$

**2. Border Point**: Within $\epsilon$ of core point, but not core itself

**3. Noise Point**: Neither core nor border

**Density-Reachable**: Point $p$ is density-reachable from $q$ if there's a chain of core points

### 9.4.2 DBSCAN Algorithm

```
Algorithm: DBSCAN(Data D, epsilon ε, minPts)

begin
    clusters ← []
    visited ← set()
    cluster_id ← 0
    
    for each point p in D do
        if p in visited then
            continue
        
        visited.add(p)
        
        // Find neighbors
        neighbors ← {q ∈ D : ||p - q|| ≤ ε}
        
        if |neighbors| < minPts then
            mark p as noise
        else
            // Start new cluster
            cluster_id ← cluster_id + 1
            EXPAND-CLUSTER(p, neighbors, cluster_id, ε, minPts)
    
    return clusters
end

Algorithm: EXPAND-CLUSTER(p, neighbors, cluster_id, ε, minPts)

begin
    assign p to cluster_id
    queue ← neighbors
    
    while queue not empty do
        q ← queue.pop()
        
        if q not visited then
            visited.add(q)
            new_neighbors ← {r ∈ D : ||q - r|| ≤ ε}
            
            if |new_neighbors| ≥ minPts then
                queue.extend(new_neighbors)
        
        if q not assigned to any cluster then
            assign q to cluster_id
end
```

### 9.4.3 Properties

**Advantages**:
- Finds clusters of arbitrary shape
- Automatically determines number of clusters
- Handles noise (outliers) explicitly
- Does not require $k$ parameter

**Disadvantages**:
- Sensitive to $\epsilon$ and MinPts
- Difficulty with varying densities
- $O(n^2)$ time without spatial index

**Complexity**:
- **Without index**: $O(n^2)$
- **With spatial index** (KD-tree, R-tree): $O(n \log n)$

### 9.4.4 Choosing Parameters

**$\epsilon$**: Use k-distance plot
1. For each point, compute distance to $k$-th nearest neighbor
2. Sort distances in descending order
3. Plot distances
4. Choose $\epsilon$ at "elbow" (sharp change)

**MinPts**: Rule of thumb: $\text{MinPts} = 2 \times d$ where $d$ is dimensionality

## 9.5 Principal Component Analysis (PCA)

**PCA** finds orthogonal directions of maximum variance for dimensionality reduction.

### 9.5.1 Problem Formulation

**Given**: Data matrix $\mathbf{D} \in \mathbb{R}^{n \times d}$ (centered)

**Goal**: Find $k$ orthonormal directions that maximize variance

$$\max_{\mathbf{v}_1,...,\mathbf{v}_k} \sum_{i=1}^{k} \mathbf{v}_i^T \mathbf{C} \mathbf{v}_i$$

subject to $\mathbf{v}_i^T \mathbf{v}_i = 1$ and $\mathbf{v}_i^T \mathbf{v}_j = 0$ for $i \neq j$

where $\mathbf{C} = \frac{1}{n}\mathbf{D}^T\mathbf{D}$ is covariance matrix.

### 9.5.2 Solution: Eigenvalue Decomposition

**Theorem**: Principal components are eigenvectors of covariance matrix

$$\mathbf{C} \mathbf{v}_i = \lambda_i \mathbf{v}_i$$

where $\lambda_1 \geq \lambda_2 \geq ... \geq \lambda_d \geq 0$

**Variance captured** by $i$-th component: $\lambda_i$

### 9.5.3 PCA Algorithm

```
Algorithm: PCA(Data D, num_components k)

begin
    // Step 1: Center the data
μ ← mean of D (column-wise)
    D_centered ← D - μ
    
    // Step 2: Compute covariance matrix
    C ← (1/n) D_centered^T D_centered
    
    // Step 3: Eigenvalue decomposition
    (λ_1, ..., λ_d), (v_1, ..., v_d) ← EIGEN(C)
    
    // Step 4: Sort by eigenvalue (descending)
    Sort eigenvectors by λ_i in descending order
    
    // Step 5: Select top k components
    V ← [v_1, v_2, ..., v_k]  // d×k matrix
    
    // Step 6: Project data
    D_reduced ← D_centered V  // n×k matrix
    
    return V, D_reduced, μ
end
```

### 9.5.4 Choosing k

**Explained Variance Ratio**:
$$\text{EVR}(k) = \frac{\sum_{i=1}^{k} \lambda_i}{\sum_{i=1}^{d} \lambda_i}$$

**Methods**:
1. **Threshold**: Choose $k$ such that EVR$(k) \geq 0.9$ (retain 90% variance)
2. **Scree plot**: Plot eigenvalues, look for elbow
3. **Cross-validation**: Use downstream task performance

### 9.5.5 Reconstruction

**Project** to low-dimensional space:
$$\mathbf{z}_i = \mathbf{V}^T (\mathbf{x}_i - \boldsymbol{\mu})$$

**Reconstruct** approximate original:
$$\hat{\mathbf{x}}_i = \mathbf{V} \mathbf{z}_i + \boldsymbol{\mu}$$

**Reconstruction error**:
$$\text{Error} = \sum_{i=1}^{n} ||\mathbf{x}_i - \hat{\mathbf{x}}_i||^2 = \sum_{i=k+1}^{d} \lambda_i$$

### 9.5.6 Complexity

**Time**: $O(d^2 n + d^3)$
- $O(d^2 n)$: Compute covariance
- $O(d^3)$: Eigenvalue decomposition

**Space**: $O(d^2)$ for covariance matrix

**Note**: For $n \ll d$, compute $\mathbf{D}\mathbf{D}^T$ instead (costs $O(n^2 d + n^3)$)

## 9.6 Singular Value Decomposition (SVD)

**SVD** factorizes any matrix into three matrices with special properties.

### 9.6.1 SVD Theorem

Any matrix $\mathbf{D} \in \mathbb{R}^{n \times d}$ can be factorized:

$$\mathbf{D} = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^T$$

where:
- $\mathbf{U} \in \mathbb{R}^{n \times n}$: Left singular vectors (orthonormal)
- $\boldsymbol{\Sigma} \in \mathbb{R}^{n \times d}$: Diagonal matrix of singular values $\sigma_1 \geq \sigma_2 \geq ... \geq 0$
- $\mathbf{V} \in \mathbb{R}^{d \times d}$: Right singular vectors (orthonormal)

### 9.6.2 Connection to PCA

For centered data $\mathbf{D}$:
- **Right singular vectors** $\mathbf{V}$ = Principal components
- **Singular values** $\sigma_i = \sqrt{n\lambda_i}$ where $\lambda_i$ are eigenvalues

$$\mathbf{C} = \frac{1}{n}\mathbf{D}^T\mathbf{D} = \mathbf{V}\frac{\boldsymbol{\Sigma}^2}{n}\mathbf{V}^T$$

### 9.6.3 Truncated SVD

**Low-rank approximation**: Keep only top $k$ components

$$\mathbf{D} \approx \mathbf{D}_k = \mathbf{U}_k \boldsymbol{\Sigma}_k \mathbf{V}_k^T$$

where subscript $k$ means first $k$ columns/rows.

**Eckart-Young Theorem**: $\mathbf{D}_k$ is best rank-$k$ approximation:

$$\mathbf{D}_k = \arg\min_{\text{rank}(\mathbf{A})=k} ||\mathbf{D} - \mathbf{A}||_F$$

**Reconstruction error**: $||\mathbf{D} - \mathbf{D}_k||_F^2 = \sum_{i=k+1}^{r} \sigma_i^2$

### 9.6.4 Applications

**1. Dimensionality Reduction**:
$$\mathbf{Z} = \mathbf{D}\mathbf{V}_k \in \mathbb{R}^{n \times k}$$

**2. Latent Semantic Analysis** (text mining):
- Rows: Documents
- Columns: Terms
- Low-rank approximation reveals topics

**3. Recommender Systems**:
- Rows: Users
- Columns: Items
- Missing entries predicted from low-rank structure

**4. Image Compression**:
- Each image row as data point
- Low-rank approximation reduces size

## 9.7 Autoencoders

**Autoencoders** are neural networks that learn compressed representations by reconstructing inputs.

### 9.7.1 Architecture

**Goal**: Learn identity function through bottleneck

$$\mathbf{x} \xrightarrow{\text{Encoder}} \mathbf{z} \xrightarrow{\text{Decoder}} \hat{\mathbf{x}} \approx \mathbf{x}$$

**Components**:
- **Encoder**: $\mathbf{z} = f(\mathbf{W}_e \mathbf{x} + \mathbf{b}_e)$
- **Decoder**: $\hat{\mathbf{x}} = g(\mathbf{W}_d \mathbf{z} + \mathbf{b}_d)$
- **Code/Latent**: $\mathbf{z} \in \mathbb{R}^k$ with $k < d$

**Loss**: Reconstruction error
$$\mathcal{L} = \sum_{i=1}^{n} ||\mathbf{x}_i - \hat{\mathbf{x}}_i||^2$$

### 9.7.2 Linear Autoencoder

With linear activation ($f = g = \text{identity}$):

$$\mathbf{z} = \mathbf{W}_e \mathbf{x}$$
$$\hat{\mathbf{x}} = \mathbf{W}_d \mathbf{z} = \mathbf{W}_d \mathbf{W}_e \mathbf{x}$$

**Theorem**: Optimal $\mathbf{W}_e$ spans same subspace as top $k$ principal components

**Connection to PCA**:
- Linear autoencoder ≈ PCA
- Both find best linear subspace
- PCA gives orthonormal basis, autoencoder gives arbitrary basis

### 9.7.3 Nonlinear Autoencoders

**Deep Autoencoder** with nonlinear activations:

**Encoder**:
$$\begin{align}
\mathbf{h}^{(1)} &= \sigma(\mathbf{W}_1 \mathbf{x} + \mathbf{b}_1) \\
\mathbf{h}^{(2)} &= \sigma(\mathbf{W}_2 \mathbf{h}^{(1)} + \mathbf{b}_2) \\
\mathbf{z} &= \mathbf{h}^{(L/2)}
\end{align}$$

**Decoder** (symmetric):
$$\begin{align}
\mathbf{h}^{(L/2+1)} &= \sigma(\mathbf{W}_{L/2+1} \mathbf{z} + \mathbf{b}_{L/2+1}) \\
&\vdots \\
\hat{\mathbf{x}} &= \sigma(\mathbf{W}_L \mathbf{h}^{(L-1)} + \mathbf{b}_L)
\end{align}$$

**Advantages over PCA**:
- Captures nonlinear structure
- Can learn hierarchical features
- More flexible representations

### 9.7.4 Training Autoencoders

```
Algorithm: TRAIN-AUTOENCODER(Data D, architecture, epochs)

begin
    Initialize weights W randomly
    
    for epoch = 1 to epochs do
        for each batch B in D do
            // Forward pass
            z ← ENCODE(B, W_encoder)
            x_hat ← DECODE(z, W_decoder)
            
            // Compute loss
            loss ← ||B - x_hat||²
            
            // Backward pass
            gradients ← BACKPROP(loss, W)
            
            // Update weights
            W ← W - η gradients
    
    return W_encoder, W_decoder
end
```

### 9.7.5 Variants

**1. Denoising Autoencoder**:
- Add noise to input: $\tilde{\mathbf{x}} = \mathbf{x} + \boldsymbol{\epsilon}$
- Train to reconstruct clean $\mathbf{x}$
- Learns robust features

**2. Sparse Autoencoder**:
- Add sparsity penalty: $\mathcal{L} = ||\mathbf{x} - \hat{\mathbf{x}}||^2 + \lambda ||\mathbf{z}||_1$
- Encourages sparse activations

**3. Variational Autoencoder (VAE)**:
- Learns probabilistic latent space
- Can generate new samples
- Uses KL divergence in loss

## 9.8 Gaussian Mixture Models & EM

**GMM** models data as mixture of Gaussian distributions.

### 9.8.1 Mixture Model

**Probability density**:
$$p(\mathbf{x}) = \sum_{j=1}^{k} \pi_j \mathcal{N}(\mathbf{x} | \boldsymbol{\mu}_j, \boldsymbol{\Sigma}_j)$$

where:
- $k$: Number of components
- $\pi_j$: Mixing coefficient ($\sum_j \pi_j = 1$, $\pi_j \geq 0$)
- $\boldsymbol{\mu}_j$: Mean of component $j$
- $\boldsymbol{\Sigma}_j$: Covariance of component $j$

**Gaussian density**:
$$\mathcal{N}(\mathbf{x} | \boldsymbol{\mu}, \boldsymbol{\Sigma}) = \frac{1}{(2\pi)^{d/2}|\boldsymbol{\Sigma}|^{1/2}} \exp\left(-\frac{1}{2}(\mathbf{x}-\boldsymbol{\mu})^T\boldsymbol{\Sigma}^{-1}(\mathbf{x}-\boldsymbol{\mu})\right)$$

### 9.8.2 EM Algorithm for GMM

**Goal**: Maximize log-likelihood
$$\mathcal{L} = \sum_{i=1}^{n} \log p(\mathbf{x}_i)$$

**Challenge**: Cannot optimize directly (sum inside log)

**Solution**: Expectation-Maximization (EM)

```
Algorithm: EM-GMM(Data D, num_components k, max_iterations T)

begin
    // Initialize parameters
π ← [1/k, ..., 1/k]
μ ← Randomly sample k points from D
Σ ← [I, ..., I]  // Identity matrices
    
    for t = 1 to T do
        // E-step: Compute responsibilities
        for i = 1 to n do
            for j = 1 to k do
                γ_ij ← π_j N(x_i | μ_j, Σ_j) / ∑_{j'} π_{j'} N(x_i | μ_{j'}, Σ_{j'})
        
        // M-step: Update parameters
        for j = 1 to k do
            N_j ← ∑_i γ_ij
            
            // Update mixing coefficient
π_j ← N_j / n
            
            // Update mean
μ_j ← (1/N_j) ∑_i γ_ij x_i
            
            // Update covariance
Σ_j ← (1/N_j) ∑_i γ_ij (x_i - μ_j)(x_i - μ_j)^T
        
        // Check convergence
        if log-likelihood change < threshold then
            break
    
    return π, μ, Σ
end
```

### 9.8.3 EM Algorithm Intuition

**E-step** (Expectation):
- **Responsibility** $\gamma_{ij}$: Probability that point $i$ belongs to component $j$
- Computed using Bayes' rule
$$\gamma_{ij} = P(z_i = j | \mathbf{x}_i) = \frac{\pi_j \mathcal{N}(\mathbf{x}_i | \boldsymbol{\mu}_j, \boldsymbol{\Sigma}_j)}{\sum_{j'=1}^{k} \pi_{j'} \mathcal{N}(\mathbf{x}_i | \boldsymbol{\mu}_{j'}, \boldsymbol{\Sigma}_{j'})}$$

**M-step** (Maximization):
- Update parameters using weighted maximum likelihood
- Each point weighted by its responsibility

**Soft clustering**: Points can partially belong to multiple clusters

### 9.8.4 EM Convergence

**Theorem**: EM monotonically increases log-likelihood
$$\mathcal{L}^{(t+1)} \geq \mathcal{L}^{(t)}$$

**Convergence**: To local maximum (not necessarily global)

**Solution**: Run with multiple random initializations

## 9.9 Summary

### Clustering Methods

| Method | Type | Complexity | Pros | Cons |
|--------|------|------------|------|------|
| k-Means | Partitional | $O(nkd)$ per iter | Fast, simple | Spherical clusters, needs $k$ |
| Hierarchical | Hierarchical | $O(n^2 \log n)$ | Dendrogram, no $k$ | Slow, $O(n^2)$ space |
| DBSCAN | Density | $O(n^2)$ or $O(n\log n)$ | Arbitrary shapes, handles noise | Sensitive to parameters |
| GMM+EM | Probabilistic | $O(nkd)$ per iter | Soft clustering, probability | Local optima, needs $k$ |

### Dimensionality Reduction

| Method | Type | Complexity | Preserves | Use Case |
|--------|------|------------|-----------|----------|
| PCA | Linear | $O(d^2n + d^3)$ | Variance | Visualization, preprocessing |
| SVD | Linear | $O(\min(nd^2, n^2d))$ | Variance | Matrix factorization, LSA |
| Autoencoder | Nonlinear | Depends on architecture | Complex features | Deep learning, generation |

### Key Takeaways

**Clustering**:
- Choose based on cluster shape and size
- k-Means for speed, DBSCAN for arbitrary shapes
- Always try multiple initializations

**Dimensionality Reduction**:
- PCA for linear relationships
- Autoencoders for nonlinear manifolds
- Always check reconstruction error

**Best Practices**:
1. Normalize/standardize features
2. Try multiple methods and compare
3. Visualize results when possible
4. Use domain knowledge to validate
5. Consider computational constraints

## 9.10 Implementation

For complete Python implementations, see:

[ch09_unsupervised_learning_implementation.ipynb](ch09_unsupervised_learning_implementation.ipynb)

The implementation notebook includes:

**Clustering**:
1. k-Means from scratch
2. k-Means++ initialization
3. Hierarchical clustering with dendrograms
4. DBSCAN implementation
5. GMM with EM algorithm
6. Cluster evaluation metrics (silhouette, DB index)

**Dimensionality Reduction**:
1. PCA from scratch
2. SVD and truncated SVD
3. Linear autoencoder
4. Deep autoencoder with PyTorch
5. Denoising autoencoder
6. Visualization with 2D reduction

**Applications**:
1. Customer segmentation
2. Image compression
3. Document clustering
4. Anomaly detection

## Further Reading

### Textbooks

- Aggarwal, C. C. (2021). *Artificial Intelligence: A Textbook*. Springer. [Chapter 9]
- Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*. Springer. [Chapters 9, 12]
- Murphy, K. P. (2012). *Machine Learning: A Probabilistic Perspective*. MIT Press.

### Classic Papers

**Clustering**:
- Lloyd, S. (1982). Least squares quantization in PCM. *IEEE Transactions on Information Theory*.
- Ester, M., et al. (1996). A density-based algorithm for discovering clusters. *KDD*.

**Dimensionality Reduction**:
- Pearson, K. (1901). On lines and planes of closest fit to systems of points in space. *Philosophical Magazine*.
- Hinton, G. E., & Salakhutdinov, R. R. (2006). Reducing the dimensionality of data with neural networks. *Science*.

**EM Algorithm**:
- Dempster, A. P., Laird, N. M., & Rubin, D. B. (1977). Maximum likelihood from incomplete data via the EM algorithm. *Journal of the Royal Statistical Society*.